# Project 1 — RAG Pipeline: Build

**Duration:** 5 min

## Index Documents

In [ ]:
from datasets import load_dataset
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load 500 ArXiv abstracts
ds = load_dataset('scientific_papers', 'arxiv', split='train[:500]', trust_remote_code=True)
docs = [d['abstract'] for d in ds]

# Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.create_documents(docs)
print(f'{len(chunks)} chunks created')

# Embed with a free local model
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local('arxiv_index')

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/ai-projects-advanced/ap-project1-build.ipynb)


## Build RAG Chain

In [ ]:
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

vectorstore = FAISS.load_local('arxiv_index', embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

# Use Ollama locally (free) or swap for any OpenAI-compatible API
llm = Ollama(model='llama3.2:3b')

prompt = PromptTemplate(
    input_variables=['context','question'],
    template="""Use the following research paper excerpts to answer the question.
If the answer is not in the context, say 'I don't have enough information.'

Context:
{context}

Question: {question}
Answer:"""
)

chain = RetrievalQA.from_chain_type(
    llm=llm, retriever=retriever,
    chain_type_kwargs={'prompt': prompt},
    return_source_documents=True
)

result = chain.invoke({'query': 'What are the main approaches to attention mechanisms in transformers?'})
print(result['result'])

## Gradio UI

In [ ]:
import gradio as gr

def ask(question):
    result = chain.invoke({'query': question})
    sources = '\n\n'.join([f'[{i+1}] {d.page_content[:200]}...'
                            for i, d in enumerate(result['source_documents'][:3])])
    return result['result'], sources

gr.Interface(
    fn=ask,
    inputs=gr.Textbox(label='Ask a question about AI research'),
    outputs=[gr.Textbox(label='Answer'), gr.Textbox(label='Sources')],
    title='ArXiv RAG Q&A'
).launch()

> **💡 Tip:** ✅ Project 1 complete! You've built a production-style RAG system with a UI.